# ONG Orchid Identifier v2 — Google Colab Runner

**Workflow:**
1. Mount Google Drive
2. Install packages
3. Unzip dataset (first time only)
4. Run training scripts

**Sebelum mulai — pastikan sudah upload ke Drive:**
- `MyDrive/orchid_project/photos.zip` — dataset (28,715 foto, 127 genera)
- `MyDrive/orchid_project/data/splits/train_live.csv`
- `MyDrive/orchid_project/data/splits/val_live.csv`
- `MyDrive/orchid_project/data/splits/test_live.csv`
- `MyDrive/orchid_project/data/image_type_labels_2ndrev.csv`
- `MyDrive/orchid_project/data/bh_species_map.json` — species lookup for BH/KM/iNat photos
- `MyDrive/orchid_project/scripts/*.py` — training scripts

**Sumber foto:**
| Sumber | Foto | Suffix |
|---|---|---|
| Bird's Head Lucid | 6,086 | nama asli (`_rs`, `_efdvogel`, dll) |
| ONG website | 12,434 | `_Orchids of New Guinea` |
| Jeffrey Champion | 4,269 | `_Jeffrey Champion` |
| Kurt Metzger | 3,345 | `_Kurt Metzger` |
| iNaturalist RG | 1,283 | `_inaturalist_RG` |
| iNaturalist NID | 1,301 | `_inaturalist_NID` (genus only, excluded from FAISS) |

In [ ]:
# ── Cell 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_PROJECT = '/content/drive/MyDrive/orchid_project'

if os.path.exists(DRIVE_PROJECT):
    print('Drive mounted. Project folder:', DRIVE_PROJECT)
    print('Contents:', os.listdir(DRIVE_PROJECT))
else:
    print('ERROR: Folder not found:', DRIVE_PROJECT)
    print('Pastikan folder di Drive bernama "orchid_project" (underscore, bukan hyphen)')

In [ ]:
# ── Cell 2: Check GPU ────────────────────────────────────────────────────────
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

!nvidia-smi

In [ ]:
# ── Cell 3: Install extra packages ──────────────────────────────────────────
# PyTorch & torchvision are pre-installed on Colab
# timm, faiss-gpu-cu12, fastai need manual install
# Note: faiss-gpu is deprecated — use faiss-gpu-cu12 for CUDA 12.x (Colab default)
!pip install timm>=1.0.0 fastai>=2.7.0 faiss-gpu-cu12 -q
!pip install gradio>=4.0.0 scikit-learn tqdm -q
print('Packages installed.')

In [ ]:
# ── Cell 3b: Verify required files on Drive ──────────────────────────────────
import os

required = {
    'photos.zip':                    f'{DRIVE_PROJECT}/photos.zip',
    'train_live.csv':                f'{DRIVE_PROJECT}/data/splits/train_live.csv',
    'val_live.csv':                  f'{DRIVE_PROJECT}/data/splits/val_live.csv',
    'test_live.csv':                 f'{DRIVE_PROJECT}/data/splits/test_live.csv',
    'image_type_labels_2ndrev.csv':  f'{DRIVE_PROJECT}/data/image_type_labels_2ndrev.csv',
    'bh_species_map.json':           f'{DRIVE_PROJECT}/data/bh_species_map.json',
    '03_train_efficientnet_colab.py':f'{DRIVE_PROJECT}/scripts/03_train_efficientnet_colab.py',
    '04_build_faiss_colab.py':       f'{DRIVE_PROJECT}/scripts/04_build_faiss_colab.py',
}

all_ok = True
for name, path in required.items():
    exists = os.path.exists(path)
    size   = f'{os.path.getsize(path)/1e9:.2f} GB' if exists and os.path.getsize(path) > 1e6 else ''
    status = 'OK' if exists else 'MISSING'
    print(f'  [{status}]  {name}  {size}')
    if not exists:
        all_ok = False

print()
print('All files present — ready to continue.' if all_ok else 'Upload missing files to Drive first.')

In [ ]:
# ── Cell 4: Unzip merged dataset (run ONCE — ~9 GB) ──────────────────────────
import os, shutil

PHOTOS_ZIP  = f'{DRIVE_PROJECT}/photos.zip'
PHOTOS_DEST = '/content/photos'               # Colab local SSD — faster than Drive

# Check if dataset is already properly unzipped (has actual genus folders)
existing_dirs = [d for d in os.scandir(PHOTOS_DEST) if d.is_dir()] if os.path.exists(PHOTOS_DEST) else []

if len(existing_dirs) == 0:
    # Empty or missing — (re)unzip
    if os.path.exists(PHOTOS_DEST):
        shutil.rmtree(PHOTOS_DEST)
    os.makedirs(PHOTOS_DEST, exist_ok=True)
    print('Unzipping dataset to /content/photos ...')
    !unzip -q "{PHOTOS_ZIP}" -d "{PHOTOS_DEST}"
    print('Done!')
else:
    print('Dataset already unzipped:', PHOTOS_DEST)

# Fix: if unzip created a nested photos/ folder, move contents up
nested = os.path.join(PHOTOS_DEST, 'photos')
if os.path.isdir(nested):
    for item in os.listdir(nested):
        shutil.move(os.path.join(nested, item), PHOTOS_DEST)
    os.rmdir(nested)
    print('Fixed nested folder structure.')

# Verify
items = [d for d in os.scandir(PHOTOS_DEST) if d.is_dir()]
total_imgs = sum(
    len([f for f in os.scandir(d.path) if f.is_file()])
    for d in items
)
print(f'Genera found : {len(items)}   (expected 127)')
print(f'Total images : {total_imgs:,}  (expected 28,715)')
print()
if total_imgs == 28715:
    print('Dataset OK — ready for training.')
else:
    print(f'WARNING: expected 28,715 but found {total_imgs:,}')
    print('  BH Lucid(6,086) + ONG(12,434) + JC(4,269) + KM(3,345) + iNat RG(1,283) + iNat NID(1,301)')

In [ ]:
# ── Cell 5: Reorganise dataset into genus folders (run ONCE) ─────────────────
%run {DRIVE_PROJECT}/scripts/01_reorganise_colab.py

In [ ]:
# ── Cell 6: Train ResNet50 baseline (Day 3-4) ────────────────────────────────
%run {DRIVE_PROJECT}/scripts/02_train_resnet50_colab.py

In [ ]:
# ── Cell 7: Train EfficientNet-B4 (with checkpoint resume) ──────────────────
import json, time, csv
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import accuracy_score, top_k_accuracy_score

# ── Paths ─────────────────────────────────────────────────────────────────────
SPLITS_DIR  = Path(f"{DRIVE_PROJECT}/data/splits")
MODELS_DIR  = Path(f"{DRIVE_PROJECT}/models/efficientnet_b4")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT  = MODELS_DIR / "checkpoint.pt"

# ── Config ────────────────────────────────────────────────────────────────────
IMG_SIZE     = 380
BATCH_SIZE   = 32
SEED         = 42
NUM_WORKERS  = 2
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PHASES = [
    {"name": "head_only",        "epochs": 5,  "lr": 1e-3, "unfreeze": 0},
    {"name": "partial_unfreeze", "epochs": 10, "lr": 1e-4, "unfreeze": 2},
    {"name": "full_finetune",    "epochs": 15, "lr": 5e-5, "unfreeze": -1},
]
EARLY_STOP   = 7
LABEL_SMOOTH = 0.1

torch.manual_seed(SEED); np.random.seed(SEED)
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ── Dataset ───────────────────────────────────────────────────────────────────
class OrchidDS(Dataset):
    def __init__(self, df, g2i, tfm):
        self.df, self.g2i, self.tfm = df.reset_index(drop=True), g2i, tfm
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(r["path"]).convert("RGB")
        return self.tfm(img), self.g2i[r["genus"]]

train_tfm = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomRotation(30),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
val_tfm = transforms.Compose([
    transforms.Resize(int(IMG_SIZE*1.1)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# ── Remap Windows paths → Colab /content/photos/ ─────────────────────────────
def remap_path(p: str) -> str:
    parts = p.replace("\\", "/").split("/")
    try:
        idx = [x.lower() for x in parts].index("photos")
        return "/content/" + "/".join(parts[idx:])
    except ValueError:
        return p

train_df = pd.read_csv(SPLITS_DIR/"train_live.csv")
val_df   = pd.read_csv(SPLITS_DIR/"val_live.csv")
test_df  = pd.read_csv(SPLITS_DIR/"test_live.csv")

train_df["path"] = train_df["path"].apply(remap_path)
val_df["path"]   = val_df["path"].apply(remap_path)
test_df["path"]  = test_df["path"].apply(remap_path)

# Filter val/test to genera present in train (avoids KeyError for rare genera)
train_genera = set(train_df["genus"].unique())
val_df   = val_df[val_df["genus"].isin(train_genera)]
test_df  = test_df[test_df["genus"].isin(train_genera)]

genera     = sorted(train_genera)
g2i        = {g:i for i,g in enumerate(genera)}
i2g        = {i:g for g,i in g2i.items()}
n_cls      = len(genera)
all_labels = list(range(n_cls))
print(f"Genera: {n_cls} | Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

with open(MODELS_DIR/"vocab.json","w") as f: json.dump(genera,f,indent=2)

def make_loader(df, tfm, shuffle):
    return DataLoader(OrchidDS(df,g2i,tfm), batch_size=BATCH_SIZE,
                      shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=True)

train_dl = make_loader(train_df, train_tfm, True)
val_dl   = make_loader(val_df,   val_tfm,   False)
test_dl  = make_loader(test_df,  val_tfm,   False)

# ── Model ─────────────────────────────────────────────────────────────────────
model     = timm.create_model("efficientnet_b4", pretrained=True, num_classes=n_cls).to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
scaler    = torch.amp.GradScaler("cuda") if DEVICE.type=="cuda" else None

def set_freeze(model, unfreeze):
    for p in model.parameters(): p.requires_grad = False
    for p in model.classifier.parameters(): p.requires_grad = True
    if unfreeze == -1:
        for p in model.parameters(): p.requires_grad = True
    elif unfreeze > 0:
        for blk in list(model.blocks)[-unfreeze:]:
            for p in blk.parameters(): p.requires_grad = True
        for p in model.conv_head.parameters(): p.requires_grad = True

def run_epoch(loader, train=True, optimizer=None):
    model.train() if train else model.eval()
    tot_loss = correct = total = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labs in tqdm(loader, leave=False):
            imgs, labs = imgs.to(DEVICE), labs.to(DEVICE)
            if train: optimizer.zero_grad()
            if scaler and train:
                with torch.amp.autocast("cuda"):
                    out = model(imgs); loss = criterion(out, labs)
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            else:
                out = model(imgs); loss = criterion(out, labs)
                if train: loss.backward(); optimizer.step()
            tot_loss += loss.item()*imgs.size(0)
            correct  += (out.argmax(1)==labs).sum().item()
            total    += imgs.size(0)
    return tot_loss/total, correct/total

def evaluate_topk(loader):
    model.eval()
    all_p, all_l = [], []
    with torch.no_grad():
        for imgs, labs in loader:
            all_p.append(model(imgs.to(DEVICE)).cpu())
            all_l.append(labs)
    p = torch.cat(all_p); l = torch.cat(all_l).numpy()
    top1 = accuracy_score(l, p.argmax(1).numpy())
    top5 = top_k_accuracy_score(l, p.numpy(), k=5, labels=all_labels)
    loss = criterion(p, torch.tensor(l, dtype=torch.long)).item()
    return top1, top5, loss

# ── Checkpoint: resume if available ───────────────────────────────────────────
history         = []
best_val        = 0.0
patience        = 0
start_phase_idx = 0
start_epoch     = 1

if CHECKPOINT.exists():
    ck = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck["model_state"])
    start_phase_idx = ck["phase_idx"]
    start_epoch     = ck["epoch"] + 1
    history         = ck["history"]
    best_val        = ck["best_val"]
    patience        = ck["patience"]
    phase_name      = PHASES[start_phase_idx]["name"]
    total_epochs    = PHASES[start_phase_idx]["epochs"]
    if start_epoch > total_epochs:
        start_phase_idx += 1
        start_epoch      = 1
        if start_phase_idx < len(PHASES):
            print(f"Resuming: moving to next phase → {PHASES[start_phase_idx]['name']}")
        else:
            print("All phases already completed — skipping training loop")
    else:
        print(f"Resuming from: phase={phase_name}, epoch={start_epoch}/{total_epochs}  |  best_val={best_val*100:.2f}%")
else:
    print("No checkpoint found — starting fresh")

# ── Training loop ─────────────────────────────────────────────────────────────
for phase_idx, phase in enumerate(PHASES):
    if phase_idx < start_phase_idx:
        print(f"Skipping phase '{phase['name']}' (already completed)")
        continue

    set_freeze(model, phase["unfreeze"])
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n{'='*55}\nPhase: {phase['name']} | Epochs: {phase['epochs']} | "
          f"LR: {phase['lr']} | Trainable params: {trainable:,}")

    opt = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                      lr=phase["lr"], weight_decay=1e-4)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=phase["epochs"])

    ep_start = start_epoch if phase_idx == start_phase_idx else 1
    for _ in range(ep_start - 1):
        sch.step()

    for ep in range(ep_start, phase["epochs"]+1):
        t0 = time.time()
        tr_loss, tr_acc  = run_epoch(train_dl, True, opt)
        v1, v5, val_loss = evaluate_topk(val_dl)
        sch.step()

        print(f"  [{phase['name']} {ep}/{phase['epochs']}]  "
              f"loss={tr_loss:.4f}  val_top1={v1*100:.2f}%  "
              f"val_top5={v5*100:.2f}%  {time.time()-t0:.0f}s")

        history.append({"phase":phase["name"],"epoch":ep,
                        "train_loss":round(tr_loss,4),"train_acc":round(tr_acc,4),
                        "val_loss":round(val_loss,4),"val_top1":round(v1,4),"val_top5":round(v5,4)})

        if v1 > best_val:
            best_val = v1; patience = 0
            torch.save(model.state_dict(), MODELS_DIR/"best_model.pth")
            print(f"    >> Best val: {best_val*100:.2f}% — saved to Drive")
        else:
            patience += 1
            if patience >= EARLY_STOP:
                print(f"    >> Early stopping"); break

        torch.save({
            "model_state": model.state_dict(),
            "phase_idx":   phase_idx,
            "epoch":       ep,
            "history":     history,
            "best_val":    best_val,
            "patience":    patience,
        }, CHECKPOINT)

    start_epoch = 1

# ── Save history & test eval ───────────────────────────────────────────────────
with open(MODELS_DIR/"training_history.csv","w",newline="") as f:
    w = csv.DictWriter(f, fieldnames=history[0].keys()); w.writeheader(); w.writerows(history)

model.load_state_dict(torch.load(MODELS_DIR/"best_model.pth", map_location=DEVICE, weights_only=False))
t1, t5, tloss = evaluate_topk(test_dl)
print(f"\n{'='*55}\nTEST RESULTS — Best model")
print(f"  Top-1: {t1*100:.2f}%  |  Top-5: {t5*100:.2f}%  |  Loss: {tloss:.4f}")

results = {"model":"EfficientNet-B4","best_val_top1":round(best_val*100,2),
           "test_top1":round(t1*100,2),"test_top5":round(t5*100,2),
           "genera":n_cls,"train":len(train_df),"val":len(val_df),"test":len(test_df)}
with open(MODELS_DIR/"results.json","w") as f: json.dump(results,f,indent=2)

CHECKPOINT.unlink(missing_ok=True)
print(f"Saved: {MODELS_DIR}")
print("Training complete — checkpoint deleted. Run Cell 8 (FAISS)")

In [ ]:
# ── Cell 8: Build FAISS index ───────────────────────────────────────────────
import json
import torch
import torch.nn as nn
import numpy as np
import faiss
import timm
import csv as _csv
from torchvision import transforms
from pathlib import Path
from PIL import Image
from tqdm import tqdm

PHOTOS_DIR   = Path("/content/photos")
LABELS_CSV   = Path(f"{DRIVE_PROJECT}/data/image_type_labels_2ndrev.csv")
SP_MAP_JSON  = Path(f"{DRIVE_PROJECT}/data/bh_species_map.json")
EFFNET_DIR   = Path(f"{DRIVE_PROJECT}/models/efficientnet_b4")

IMG_SIZE   = 380
BATCH_SIZE = 64
VALID_EXTS = {".jpg",".jpeg",".png",".bmp",".tiff",".tif",".webp"}
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── Load model ────────────────────────────────────────────────────────────────
with open(EFFNET_DIR/"vocab.json") as f: genera = json.load(f)
n_cls = len(genera)

model = timm.create_model("efficientnet_b4", pretrained=False, num_classes=n_cls)
model.load_state_dict(torch.load(EFFNET_DIR/"best_model.pth", map_location=DEVICE))
model.classifier = nn.Identity()
model = model.to(DEVICE).eval()
print(f"Model loaded (embedding dim=1792, genera={n_cls})")

# ── Transform ─────────────────────────────────────────────────────────────────
tfm = transforms.Compose([
    transforms.Resize(int(IMG_SIZE*1.1)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# ── Load species map ──────────────────────────────────────────────────────────
sp_lookup = {}
if SP_MAP_JSON.exists():
    with open(SP_MAP_JSON, encoding="utf-8") as f:
        sp_lookup = json.load(f)
    print(f"Species map loaded: {len(sp_lookup):,} entries")

def parse_species(filename: str) -> str:
    if filename in sp_lookup:
        return sp_lookup[filename]
    stem  = Path(filename).stem
    parts = stem.split("_")
    while parts and not parts[-1].isdigit():
        parts = parts[:-1]
    if parts and parts[-1].isdigit():
        parts = parts[:-1]
    return " ".join(parts[1:])

# ── Load live-only labels ─────────────────────────────────────────────────────
def path_to_filename(p: str) -> str:
    return p.replace("\\", "/").split("/")[-1]

live_paths = set()
if LABELS_CSV.exists():
    with open(LABELS_CSV, encoding="utf-8") as f:
        for row in _csv.DictReader(f):
            if row["label"] in ("live", "uncertain"):
                fn = path_to_filename(row["path"])
                if "_NID" not in fn:
                    live_paths.add(fn)
    print(f"Live filter loaded: {len(live_paths):,} photos (NID excluded)")
else:
    print("WARNING: image_type_labels_2ndrev.csv not found — indexing ALL non-NID photos")

# ── Collect images ────────────────────────────────────────────────────────────
records = []
for genus_dir in sorted(PHOTOS_DIR.iterdir()):
    if not genus_dir.is_dir(): continue
    genus = genus_dir.name
    for f in genus_dir.iterdir():
        if not (f.is_file() and f.suffix.lower() in VALID_EXTS): continue
        if "_NID" in f.name: continue
        if live_paths and f.name not in live_paths: continue
        records.append({"path": str(f), "genus": genus,
                        "species": parse_species(f.name), "filename": f.name})
print(f"Images to index: {len(records):,}")

# ── Extract embeddings ────────────────────────────────────────────────────────
embeddings, valid_recs = [], []
batch_imgs, batch_recs = [], []

def flush_batch():
    if not batch_imgs: return
    with torch.no_grad():
        t = torch.stack(batch_imgs).to(DEVICE)
        e = model(t).cpu().numpy()
    embeddings.extend(e)
    valid_recs.extend(batch_recs)
    batch_imgs.clear(); batch_recs.clear()

for rec in tqdm(records, desc="Extracting"):
    try:
        img = Image.open(rec["path"]).convert("RGB")
        batch_imgs.append(tfm(img))
        batch_recs.append(rec)
    except Exception as ex:
        print(f"  Skipping {rec['path']}: {ex}"); continue
    if len(batch_imgs) == BATCH_SIZE:
        flush_batch()
flush_batch()
print(f"Embeddings: {len(embeddings):,}")

# ── Build FAISS index ─────────────────────────────────────────────────────────
arr = np.array(embeddings, dtype="float32")
faiss.normalize_L2(arr)

idx = faiss.IndexFlatIP(arr.shape[1])
idx.add(arr)
print(f"FAISS index: {idx.ntotal:,} vectors, dim={arr.shape[1]}")

# ── Save to Drive ─────────────────────────────────────────────────────────────
DRIVE_MODELS = Path(f"{DRIVE_PROJECT}/models")
faiss.write_index(idx, str(DRIVE_MODELS/"ong_species_index.faiss"))
with open(DRIVE_MODELS/"ong_metadata.json","w",encoding="utf-8") as f:
    json.dump(valid_recs, f, ensure_ascii=False)

print(f"Saved: {DRIVE_MODELS/'ong_species_index.faiss'}")
print(f"Saved: {DRIVE_MODELS/'ong_metadata.json'}")

# ── Sanity check ──────────────────────────────────────────────────────────────
print("\nSanity check (first image):")
D, I = idx.search(arr[:1], k=6)
for rank, (score, i) in enumerate(zip(D[0], I[0]), 1):
    print(f"  Top-{rank}: {valid_recs[i]['species']}  (score={score:.4f})")
print("\nDone.")

---

## Test the Model

**Cell 9** — genus classifier only (run after Cell 7, no FAISS needed)

**Cell 10** — full pipeline: genus + FAISS species retrieval (run after Cell 8)

In [ ]:
# -- Cell 9: Quick Test -- Genus Classifier ----------------------------------
# Run AFTER Cell 7 (training finished). No FAISS needed.
import torch, json, io
import torch.nn.functional as F
import timm
from torchvision import transforms
from PIL import Image
from pathlib import Path
from google.colab import files
import IPython.display as display

EFFNET_DIR = Path(f"{DRIVE_PROJECT}/models/efficientnet_b4")
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open(EFFNET_DIR/"vocab.json") as f:
    genera = json.load(f)
n_cls = len(genera)

model_cls = timm.create_model("efficientnet_b4", pretrained=False, num_classes=n_cls)
model_cls.load_state_dict(torch.load(EFFNET_DIR/"best_model.pth",
                                      map_location=DEVICE, weights_only=False))
model_cls = model_cls.to(DEVICE).eval()
print(f"Model loaded -- {n_cls} genera | Device: {DEVICE}")

tfm = transforms.Compose([
    transforms.Resize(int(380*1.1)),
    transforms.CenterCrop(380),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

print("\nUpload a photo to identify:")
uploaded = files.upload()

for fname, data in uploaded.items():
    img = Image.open(io.BytesIO(data)).convert("RGB")
    display.display(img.resize((300, 300)))
    with torch.no_grad():
        logits = model_cls(tfm(img).unsqueeze(0).to(DEVICE))
        probs  = F.softmax(logits, dim=1)[0]
    top5_p, top5_i = probs.topk(5)
    print(f"\nTop-5 genus predictions: {fname}")
    print("-" * 45)
    for p, i in zip(top5_p, top5_i):
        bar = "#" * int(p.item() * 30)
        print(f"  {genera[i.item()]:<28} {p.item()*100:5.1f}%  {bar}")


---

In [ ]:
# -- Cell 10: Full Pipeline Test -- Top-3 Genus x FAISS -> Top-5 Species x 5 Photos
# Each species shows its 5 most visually similar photos. Total: up to 25 photos.
# Run AFTER Cell 8 (FAISS index built).
import torch, json, io
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import timm, faiss
from torchvision import transforms
from PIL import Image
from pathlib import Path
from collections import defaultdict
from google.colab import files
import IPython.display as display

EFFNET_DIR   = Path(f"{DRIVE_PROJECT}/models/efficientnet_b4")
DRIVE_MODELS = Path(f"{DRIVE_PROJECT}/models")
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open(EFFNET_DIR/"vocab.json") as f:
    genera = json.load(f)
n_cls = len(genera)

model = timm.create_model("efficientnet_b4", pretrained=False, num_classes=n_cls)
model.load_state_dict(torch.load(EFFNET_DIR/"best_model.pth",
                                  map_location=DEVICE, weights_only=False))
model = model.to(DEVICE).eval()

faiss_index = faiss.read_index(str(DRIVE_MODELS/"ong_species_index.faiss"))
with open(DRIVE_MODELS/"ong_metadata.json", encoding="utf-8") as f:
    metadata = json.load(f)

genus_to_idx = defaultdict(list)
for i, rec in enumerate(metadata):
    genus_to_idx[rec["genus"]].append(i)

print(f"Model: {n_cls} genera | FAISS: {faiss_index.ntotal:,} vectors")

tfm = transforms.Compose([
    transforms.Resize(int(380*1.1)),
    transforms.CenterCrop(380),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

def remap(path):
    parts = path.replace("\\\\", "/").replace("\\", "/").split("/")
    try:
        pi = [x.lower() for x in parts].index("photos")
        return "/content/" + "/".join(parts[pi:])
    except ValueError:
        return path

def identify(img, top_n_species=5, photos_per_species=5):
    """
    Two-stage identification.
    Returns: (top5_species_results, genus_probs_top5)
    Each species result: {species, genus, best_score, genus_conf, photos: [list of {path, score}]}
    """
    tensor = tfm(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(tensor)
        probs  = F.softmax(logits, dim=1)[0]
        top3_p, top3_i = probs.topk(3)
        top5_p, top5_i = probs.topk(5)
        orig_head        = model.classifier
        model.classifier = nn.Identity()
        emb              = model(tensor).cpu().numpy().astype("float32")
        model.classifier = orig_head
    faiss.normalize_L2(emb)

    # Search top-3 genera, gather candidates per species
    from collections import defaultdict
    species_pool = defaultdict(list)

    for g_prob, g_idx in zip(top3_p.tolist(), top3_i.tolist()):
        genus     = genera[g_idx]
        g_indices = genus_to_idx.get(genus, [])
        if not g_indices:
            continue
        g_vecs  = np.array([faiss_index.reconstruct(i) for i in g_indices], dtype="float32")
        sub_idx = faiss.IndexFlatIP(g_vecs.shape[1])
        sub_idx.add(g_vecs)
        k       = min(50, len(g_indices))   # fetch more to get 5 photos/species
        D, I    = sub_idx.search(emb, k)
        for score, local_i in zip(D[0], I[0]):
            rec = metadata[g_indices[local_i]]
            combined = g_prob * float(score)
            species_pool[rec["species"]].append({
                "genus":      genus,
                "genus_conf": g_prob,
                "score":      combined,
                "faiss_score":float(score),
                "path":       rec["path"],
            })

    # For each species: sort photos by combined score, keep top-5
    for sp in species_pool:
        species_pool[sp].sort(key=lambda x: x["score"], reverse=True)
        species_pool[sp] = species_pool[sp][:photos_per_species]

    # Rank species by their best photo score, take top-5
    ranked = sorted(species_pool.items(),
                    key=lambda kv: kv[1][0]["score"], reverse=True)[:top_n_species]

    results = []
    for sp, photos in ranked:
        results.append({
            "species":    sp,
            "genus":      photos[0]["genus"],
            "genus_conf": photos[0]["genus_conf"],
            "best_score": photos[0]["score"],
            "photos":     photos,
        })

    genus_labels = {genera[i.item()]: float(p.item()) for p, i in zip(top5_p, top5_i)}
    return results, genus_labels

# Upload & run
print("\nUpload a photo:")
uploaded = files.upload()

for fname, data in uploaded.items():
    img = Image.open(io.BytesIO(data)).convert("RGB")
    results, genus_labels = identify(img)

    print(f"\n{'='*60}")
    display.display(img.resize((260, 260)))
    print(f"File: {fname}")

    # Stage 1
    print(f"\nSTAGE 1 -- Genus Predictions:")
    print("-" * 48)
    for genus, conf in sorted(genus_labels.items(), key=lambda x: -x[1]):
        bar = "#" * int(conf * 28)
        print(f"  {genus:<28} {conf*100:5.1f}%  {bar}")

    # Stage 2
    print(f"\nSTAGE 2 -- Top-5 Species (5 photos each = up to 25 total):")
    for rank, res in enumerate(results, 1):
        n_photos = len(res["photos"])
        print(f"\n  {rank}. {res['species']}")
        print(f"     Genus: {res['genus']} ({res['genus_conf']*100:.0f}%)  |  Best score: {res['best_score']:.3f}  |  {n_photos} photos")
        print(f"  {'-'*52}")
        for j, photo in enumerate(res["photos"], 1):
            print(f"    Photo {j}/{n_photos}  score={photo['score']:.3f}")
            try:
                thumb = Image.open(remap(photo["path"])).convert("RGB").resize((160,160))
                display.display(thumb)
            except:
                print("    [thumbnail not available]")


---

In [ ]:
# -- Cell 11: Gradio App -- Top-5 Species x 5 Photos Gallery -----------------
# Gallery shows up to 25 photos (5 species x 5 most-similar photos each).
# Run AFTER Cell 8 (FAISS index built).
import torch, json, io
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import timm, faiss, gradio as gr
from torchvision import transforms
from PIL import Image
from pathlib import Path
from collections import defaultdict

EFFNET_DIR   = Path(f"{DRIVE_PROJECT}/models/efficientnet_b4")
DRIVE_MODELS = Path(f"{DRIVE_PROJECT}/models")
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open(EFFNET_DIR/"vocab.json") as f:
    genera = json.load(f)
n_cls = len(genera)

model = timm.create_model("efficientnet_b4", pretrained=False, num_classes=n_cls)
model.load_state_dict(torch.load(EFFNET_DIR/"best_model.pth",
                                  map_location=DEVICE, weights_only=False))
model = model.to(DEVICE).eval()

faiss_index = faiss.read_index(str(DRIVE_MODELS/"ong_species_index.faiss"))
with open(DRIVE_MODELS/"ong_metadata.json", encoding="utf-8") as f:
    metadata = json.load(f)

genus_to_idx = defaultdict(list)
for i, rec in enumerate(metadata):
    genus_to_idx[rec["genus"]].append(i)

print(f"Model: {n_cls} genera | FAISS: {faiss_index.ntotal:,} vectors")

tfm = transforms.Compose([
    transforms.Resize(int(380*1.1)),
    transforms.CenterCrop(380),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

def remap(path):
    parts = path.replace("\\\\", "/").replace("\\", "/").split("/")
    try:
        pi = [x.lower() for x in parts].index("photos")
        return "/content/" + "/".join(parts[pi:])
    except ValueError:
        return path

def identify(img, top_n_species=5, photos_per_species=5):
    """
    Two-stage identification.
    Returns: (top5_species_results, genus_probs_top5)
    Each species result: {species, genus, best_score, genus_conf, photos: [list of {path, score}]}
    """
    tensor = tfm(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(tensor)
        probs  = F.softmax(logits, dim=1)[0]
        top3_p, top3_i = probs.topk(3)
        top5_p, top5_i = probs.topk(5)
        orig_head        = model.classifier
        model.classifier = nn.Identity()
        emb              = model(tensor).cpu().numpy().astype("float32")
        model.classifier = orig_head
    faiss.normalize_L2(emb)

    # Search top-3 genera, gather candidates per species
    from collections import defaultdict
    species_pool = defaultdict(list)

    for g_prob, g_idx in zip(top3_p.tolist(), top3_i.tolist()):
        genus     = genera[g_idx]
        g_indices = genus_to_idx.get(genus, [])
        if not g_indices:
            continue
        g_vecs  = np.array([faiss_index.reconstruct(i) for i in g_indices], dtype="float32")
        sub_idx = faiss.IndexFlatIP(g_vecs.shape[1])
        sub_idx.add(g_vecs)
        k       = min(50, len(g_indices))   # fetch more to get 5 photos/species
        D, I    = sub_idx.search(emb, k)
        for score, local_i in zip(D[0], I[0]):
            rec = metadata[g_indices[local_i]]
            combined = g_prob * float(score)
            species_pool[rec["species"]].append({
                "genus":      genus,
                "genus_conf": g_prob,
                "score":      combined,
                "faiss_score":float(score),
                "path":       rec["path"],
            })

    # For each species: sort photos by combined score, keep top-5
    for sp in species_pool:
        species_pool[sp].sort(key=lambda x: x["score"], reverse=True)
        species_pool[sp] = species_pool[sp][:photos_per_species]

    # Rank species by their best photo score, take top-5
    ranked = sorted(species_pool.items(),
                    key=lambda kv: kv[1][0]["score"], reverse=True)[:top_n_species]

    results = []
    for sp, photos in ranked:
        results.append({
            "species":    sp,
            "genus":      photos[0]["genus"],
            "genus_conf": photos[0]["genus_conf"],
            "best_score": photos[0]["score"],
            "photos":     photos,
        })

    genus_labels = {genera[i.item()]: float(p.item()) for p, i in zip(top5_p, top5_i)}
    return results, genus_labels

def predict(img):
    if img is None:
        return {}, [], ""

    results, genus_labels = identify(img)

    # Build gallery: 5 species x up to 5 photos, grouped with caption
    gallery = []
    for rank, res in enumerate(results, 1):
        for j, photo in enumerate(res["photos"], 1):
            caption = (f"#{rank} {res['species']}  [{j}/{len(res['photos'])}]\n"
                       f"Genus: {res['genus']} ({res['genus_conf']*100:.0f}%)  |  "
                       f"Score: {photo['score']:.3f}")
            try:
                pil_img = Image.open(remap(photo["path"])).convert("RGB")
            except:
                pil_img = Image.new("RGB", (150,150), (220,220,220))
            gallery.append((pil_img, caption))

    best   = results[0] if results else None
    summary = (f"Best match: {best['species']}  (score {best['best_score']:.3f})"
               if best else "No results")

    return genus_labels, gallery, summary

with gr.Blocks(title="ONG Orchid Identifier") as demo:
    gr.Markdown(
        "# ONG Orchid Identifier\n"
        "Upload an orchid photo. Stage 1 predicts genus; "
        "Stage 2 returns the **5 most similar species** — each shown with up to **5 matching photos** "
        "ranked by visual similarity (top-3 genera searched)."
    )
    with gr.Row():
        with gr.Column(scale=1):
            img_input = gr.Image(type="pil", label="Upload Orchid Photo")
            run_btn   = gr.Button("Identify", variant="primary")
        with gr.Column(scale=2):
            genus_out   = gr.Label(num_top_classes=5, label="Stage 1 — Genus Prediction")
            summary_out = gr.Textbox(label="Best Match", interactive=False)

    gallery_out = gr.Gallery(
        label="Stage 2 — Top-5 Species x 5 Photos  (25 photos total, sorted by similarity)",
        columns=5, rows=5, height=600, object_fit="cover"
    )

    run_btn.click(fn=predict, inputs=img_input,
                  outputs=[genus_out, gallery_out, summary_out])
    img_input.change(fn=predict, inputs=img_input,
                     outputs=[genus_out, gallery_out, summary_out])

demo.launch(share=True, debug=False)


In [ ]:
# ── Cell 9: Check saved models ───────────────────────────────────────────────
import os
models_dir = f'{DRIVE_PROJECT}/models'
for root, dirs, files in os.walk(models_dir):
    for f in files:
        fpath = os.path.join(root, f)
        size_mb = os.path.getsize(fpath) / 1e6
        print(f'{fpath.replace(models_dir, "")}  ({size_mb:.1f} MB)')